In [1]:
!pip install transformers datasets sentencepiece --quiet

In [2]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Trainer, TrainingArguments, DataCollatorForSeq2Seq

In [3]:
from datasets import Dataset

In [4]:
import pandas as pd

In [6]:
df = pd.read_csv("/content/neplai_roman_pairs.csv")

df = df.dropna(subset=["nepali_word", "english_word"])

# Also remove rows with just whitespace
df = df[df["nepali_word"].str.strip() != ""]
df = df[df["english_word"].str.strip() != ""]

# Reset index
df = df.reset_index(drop=True)

dataset = Dataset.from_pandas(df)


In [7]:
print(df)

            nepali_word    english_word
0              परिस्कृत       pariskrit
1                  दिशा           disha
2       गोर्खाल्याण्डका    gorkhalandka
3            चञ्चलताहरू  chanchaltaharu
4                  रूखो           rukho
...                 ...             ...
324945        तामक्रमले       tamkramle
324946           नातोमा          natoma
324947         सल्तानले        saltanle
324948      पशुपतिजस्तो  pashupatijasto
324949     फोटोस्फेरबाट              ph

[324950 rows x 2 columns]


In [8]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")

model = MBartForConditionalGeneration.from_pretrained(model_name)

# mBART requires language codes
src_lang = "en_XX"  # Romanized treated as English
tgt_lang = "ne_NP"  # Nepali

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [9]:
def preprocess(examples):
    inputs = examples["english_word"]  # Romanized input
    targets = examples["nepali_word"]  # Target Nepali

    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True,
        text_target=targets,  # Use text_target for target tokenization
    )

    # The labels for sequence-to-sequence models are the tokenized target sequence
    model_inputs["labels"] = tokenizer(
         targets, max_length=64, truncation=True
    )["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/324950 [00:00<?, ? examples/s]

In [10]:
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split['train']
eval_dataset = split['test']


In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="./mbart_translit_demo",
    eval_strategy="epoch",  # Changed from evaluation_strategy to eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    data_collator=data_collator, # Add data collator here
)

In [3]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")

model = MBartForConditionalGeneration.from_pretrained(model_name)

# mBART requires language codes
src_lang = "en_XX"  # Romanized treated as English
tgt_lang = "ne_NP"  # Nepali

In [ ]:
trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 078bct043 (077bct066-techzhub) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [1]:
def transliterate(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=64, truncation=True)
    generated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang])
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

print(transliterate("namaste"))


NameError: name 'tokenizer' is not defined